In [1]:
!rm -rf /content/drive/MyDrive/visiondoc-runs/data-cache /content/visiondoc-ai/data/cache

# VisionDoc AI — Reproduce the Published Results

**This notebook is the source of every number in the README.** Run it top to bottom on a
free Google Colab **T4** and it produces `reports/results.json`, the canonical artifact that
`scripts/update_readme.py` renders into the README table. No number reaches the README by
any other path: CI runs `make readme-check`, which regenerates the table from that JSON and
fails the build if a single character was hand-edited.

> **Runtime → Change runtime type → T4 GPU** before you start.

## The experiment

| | |
|---|---|
| **Dataset** | CORD-v2 (`naver-clova-ix/cord-v2`) — 800 train / 100 validation / 100 test receipts. It is the only corpus in this repo with a real train split that fits the free tier. |
| **Model** | `Qwen/Qwen2.5-VL-3B-Instruct`, 4-bit QLoRA, **fp16** (a T4 is Turing — it has no bf16 units at all). |
| **Training** | 2 epochs over all 800 receipts, effective batch 8, LoRA on the language side only. |
| **Eval set** | `validation + test` pooled into **one fixed 200-sample list** — the *same* list, the *same* seed and the *same* decoding parameters for both arms. Anything else is not a comparison. |
| **Primary metric** | **field-level precision / recall / F1** (`evaluation.metrics.structured_field_metrics`). |
| **Secondary metrics** | exact match, ANLS, token-F1 — reported, but clearly labelled. |
| **Statistics** | every quality metric is published with *n* and a **bootstrap 95% confidence interval**. |

### Why field-F1 is the headline and EM/ANLS/token-F1 are not

A CORD target is a *serialized JSON receipt*, not a short answer span. Exact match over a
whole JSON blob is near-zero by construction — one wrong price in a 40-field document scores
the same as total nonsense — and ANLS/token-F1 inherit most of that pathology. Field-level
P/R/F1 compares the parsed key→value pairs, which is the thing the task actually asks for.
The secondary metrics stay in the report precisely so nobody has to take that on trust.

### What to honestly expect

The base Qwen2.5-VL-3B has never seen CORD's schema, so it tends to answer in prose or in a
*differently shaped* JSON; its field-F1 should be low and the fine-tuned delta should
therefore be large. **That is a prediction, not a promise.** Whatever comes out of this
notebook is what gets published — including a small delta, an overlapping confidence
interval, or a regression — with the explanation attached. Numbers are never hand-typed,
re-rolled until they look good, or quietly dropped.

### Session deaths

Free Colab disconnects without warning. Every cell below is written to be **re-run from the
top after a disconnect**: Drive holds the HF cache, the checkpoints and the outputs, and the
training cell resumes from the newest checkpoint instead of starting over.

In [2]:
# --- 1. GPU check ----------------------------------------------------------
# Fail here, loudly, rather than 40 minutes into a CPU-only "training" run.
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU visible. Runtime > Change runtime type > T4 GPU, then Run all again."
)
NAME = torch.cuda.get_device_name(0)
MAJOR, MINOR = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU            : {NAME} (sm_{MAJOR}{MINOR}, {VRAM_GB:.1f} GB)")
print(f"torch          : {torch.__version__} (CUDA {torch.version.cuda})")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()}")

# A T4 is sm_75: no bf16. configs/benchmark_cord.yaml sets fp16 true / bf16 false
# for exactly this reason. If you are on an A100/L4 you may flip that, but then
# you are no longer reproducing the published run bit-for-bit.
if (MAJOR, MINOR) < (8, 0):
    print("\nTuring or older -> fp16 path (bf16 would crash). This matches the published config.")
else:
    print("\nAmpere+ detected. The published numbers came from a T4 in fp16; keep the config as-is to match.")

Mon Sep 14 11:20:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Persist everything to Google Drive first

A free Colab VM is deleted when the session drops, and with it ~7 GB of downloaded model
weights plus every checkpoint. So before anything else we mount Drive and point the three
things that are expensive to recreate at it:

* `HF_HOME` → the Hugging Face model + dataset cache,
* `outputs/` → checkpoints and the trained adapter,
* `data/cache/` → the preprocessed CORD splits.

The cell below performs a **real mount** and will pop up an authorization prompt the first
time. If you want a throwaway run and do not care about losing a disconnected session, set
`PERSIST_DIR=/content` in the cell (or `os.environ["PERSIST_DIR"] = "/content"` before running
it) and no Drive mount happens.

Trade-off worth knowing: reading the model cache from Drive is slower than from local disk.
That cost is paid once per session and is far cheaper than re-downloading 7 GB after every
disconnect.

In [3]:
# --- 2. Mount Drive and route the expensive artifacts there -----------------
import os
import shutil
from pathlib import Path

# Set PERSIST_DIR=/content for an ephemeral run (nothing survives a disconnect).
PERSIST_DIR = Path(os.environ.get("PERSIST_DIR", "/content/drive/MyDrive/visiondoc-runs"))

if str(PERSIST_DIR).startswith("/content/drive"):
    from google.colab import drive  # Colab-only import; guarded by the branch.

    drive.mount("/content/drive")  # real mount: expect an auth prompt the first time
else:
    print(f"EPHEMERAL RUN: PERSIST_DIR={PERSIST_DIR} is not on Drive. A disconnect loses everything.")

# Persist ONLY what is irreplaceable. Checkpoints/adapter cost GPU-hours and the
# results are the whole point, so those go to Drive. The HF model+dataset cache
# (~10 GB) is freely re-downloadable in minutes, and a free Drive is only 15 GB --
# parking it there would nearly fill the account AND make every weight load crawl
# through Drive's FUSE mount. So it stays on the VM's fast local disk by default.
# Set PERSIST_HF=1 if you would rather trade Drive space for skipping re-downloads.
# >>> FLIP THIS to True if your Drive has room to spare (a paid plan). It keeps the
# ~10 GB of model/dataset downloads on Drive so a disconnect does NOT re-download
# them, and skips rebuilding the processed dataset cache. Default False is the safe
# choice for a free 15 GB Drive. (Env var PERSIST_HF=1 does the same thing.)
PERSIST_HF = os.environ.get("PERSIST_HF") == "1"

SCRATCH_DIR = Path("/content/vd-scratch")
_hf_root = PERSIST_DIR if PERSIST_HF else SCRATCH_DIR

HF_DIR = _hf_root / "hf"                    # model + dataset downloads (local by default)
DATA_CACHE_DIR = _hf_root / "data-cache"    # rebuildable processed cache
OUTPUTS_DIR = PERSIST_DIR / "outputs"       # checkpoints + adapter  <- always persisted
REPORTS_DIR = PERSIST_DIR / "reports"       # final artifacts        <- always persisted
for d in (HF_DIR, OUTPUTS_DIR, DATA_CACHE_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# HF_HOME covers hub downloads, datasets and the processor cache in one variable.
os.environ["HF_HOME"] = str(HF_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_DIR / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_DIR / "datasets")
os.environ["PERSIST_DIR"] = str(PERSIST_DIR)  # so re-running this cell is idempotent

REPO_DIR = Path("/content/visiondoc-ai")


def _link_dir(link: Path, target: Path) -> None:
    """Point ``link`` (inside the repo) at ``target`` (on Drive), preserving content.

    Why a symlink instead of just editing the config: the published config must stay
    byte-identical to what is committed, otherwise the run is not the documented run.
    Redirecting the path at the filesystem level keeps `outputs/...` in the config true
    while the bytes actually land on Drive.
    """
    target.mkdir(parents=True, exist_ok=True)
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() == target.resolve():
            return
        link.unlink()
    elif link.exists():
        # A real directory from a previous (non-persisted) run: migrate it once
        # rather than silently orphaning whatever is inside it.
        for item in link.iterdir():
            dest = target / item.name
            if not dest.exists():
                shutil.move(str(item), str(dest))
        link.rmdir()
    link.symlink_to(target, target_is_directory=True)
    print(f"{link} -> {target}")


def link_persistent(repo_dir: Path = REPO_DIR) -> None:
    """Redirect the repo's outputs/ and data/cache/ onto Drive. Called after the clone."""
    _link_dir(repo_dir / "outputs", OUTPUTS_DIR)
    _link_dir(repo_dir / "data" / "cache", DATA_CACHE_DIR)


print(f"PERSIST_DIR = {PERSIST_DIR}   (checkpoints + results -> survives a disconnect)")
print(f"HF_HOME     = {os.environ['HF_HOME']}")
if PERSIST_HF:
    print("PERSIST_HF=True: weights + dataset cache live on Drive (~12 GB), so a")
    print("reconnect skips the re-download. Best choice on a paid Drive plan.")
else:
    print("PERSIST_HF=False: HF cache on LOCAL disk (~1 GB of Drive used). A")
    print("disconnect costs a ~7.5 GB re-download. Flip PERSIST_HF=True if you")
    print("have Drive space -- it is the faster option across multiple sessions.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PERSIST_DIR = /content/drive/MyDrive/visiondoc-runs   (checkpoints + results -> survives a disconnect)
HF_HOME     = /content/vd-scratch/hf
PERSIST_HF=False: HF cache on LOCAL disk (~1 GB of Drive used). A
disconnect costs a ~7.5 GB re-download. Flip PERSIST_HF=True if you
have Drive space -- it is the faster option across multiple sessions.


In [4]:
# --- 3. Clone (or update) the repo, then redirect its artifact dirs to Drive -
import subprocess
import sys

REPO_URL = "https://github.com/Swapnil-byte-798/visiondoc-ai.git"

if (REPO_DIR / ".git").is_dir():
    # Re-running after a disconnect: fast-forward only, so a dirty checkout is a
    # loud failure instead of a silent merge that changes what you are measuring.
    print(subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/visiondoc-ai
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))  # absolute imports rooted at the repo

link_persistent(REPO_DIR)

print("\nHEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())
CONFIG_PATH = "configs/benchmark_cord.yaml"
assert Path(CONFIG_PATH).exists(), f"{CONFIG_PATH} missing — is this the right branch?"

Updating 2ca1c6f..3caea7a
Fast-forward
 configs/benchmark_cord.yaml         |  8 +++--
 tests/test_no_eager_image_decode.py | 65 ++++++++++++++++++++++++++++++-------
 training/train.py                   | 63 +++++++++++++++++++++++++----------
 3 files changed, 104 insertions(+), 32 deletions(-)

/content/visiondoc-ai
/content/visiondoc-ai/data/cache -> /content/vd-scratch/data-cache

HEAD: 3caea7a


In [5]:
# --- 4. Pinned dependencies -------------------------------------------------
# Every package is pinned with '==' on purpose: a floating transformers/peft is
# enough to move the third decimal place, and then the published numbers are not
# reproducible. If you must bump a pin, re-run the whole notebook and republish.
#
# torch/torchvision are NOT installed here. Colab ships a CUDA build matched to
# its driver; pip-installing torch would drag in a wheel that may not match and
# can silently fall back to CPU.
PINS = [
    "transformers==4.51.3",
    "accelerate==1.6.0",
    "peft==0.15.2",
    "datasets==3.5.0",
    "bitsandbytes==0.45.5",
    "qwen-vl-utils==0.0.11",
    "evaluate==0.4.3",
    "rouge-score==0.1.2",
    "sacrebleu==2.5.1",
    "Levenshtein==0.27.1",
    "pymupdf==1.25.5",
    "pytesseract==0.3.13",
]

!pip install -q {" ".join(PINS)}

# Print what actually got resolved. The pins above are the intent; this is the
# evidence, and it is what belongs in a bug report if the numbers ever move.
from importlib.metadata import PackageNotFoundError, version

print("\nResolved versions")
print("-----------------")
import torch

print(f"{'torch':<16} {torch.__version__}  (CUDA {torch.version.cuda})")
for spec in PINS:
    dist = spec.split("==")[0]
    try:
        print(f"{dist:<16} {version(dist)}")
    except PackageNotFoundError:
        print(f"{dist:<16} NOT INSTALLED  <-- investigate before trusting any number")


Resolved versions
-----------------
torch            2.11.0+cu128  (CUDA 12.8)
transformers     4.51.3
accelerate       1.6.0
peft             0.15.2
datasets         3.5.0
bitsandbytes     0.45.5
qwen-vl-utils    0.0.11
evaluate         0.4.3
rouge-score      0.1.2
sacrebleu        2.5.1
Levenshtein      0.27.1
pymupdf          1.25.5
pytesseract      0.3.13


In [6]:
# --- 5. Seeds, offline tracking, and the resolved config --------------------
import os

# Nothing in this run may block on or phone home to a tracking backend. The
# config already sets report_to: none; this is belt-and-braces for any library
# that initialises W&B on import.
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # avoids a fork warning storm

import yaml

from configs import load_config
from utils.seed import set_seed

config = load_config(CONFIG_PATH)
set_seed(config.seed)  # python / numpy / torch, incl. CUDA

print(f"=== resolved {CONFIG_PATH} ===")
print(yaml.safe_dump(config.to_dict(), sort_keys=False, default_flow_style=False))

# Guardrails that would otherwise surface as a mid-run crash or a quietly
# invalid comparison.
assert config.training.fp16 and not config.training.bf16, "T4 has no bf16"
assert config.model.load_in_4bit, "QLoRA is what makes this fit 16 GB"
assert config.training.save_steps == config.training.eval_steps, (
    "load_best_model_at_end requires save_steps == eval_steps"
)
assert config.training.save_total_limit >= 2, (
    "save_total_limit 1 can rotate away the only usable checkpoint mid-write"
)

OUTPUT_DIR = Path(config.training.output_dir)
ADAPTER_DIR = OUTPUT_DIR / "adapter"
print(f"output_dir  : {OUTPUT_DIR}  (-> {OUTPUT_DIR.resolve()})")
print(f"adapter_dir : {ADAPTER_DIR}")
print(f"seed        : {config.seed}")

=== resolved configs/benchmark_cord.yaml ===
name: visiondoc-ai-cord-benchmark
seed: 42
device: auto
output_root: outputs
report_dir: reports
model:
  model_id: Qwen/Qwen2.5-VL-3B-Instruct
  model_type: qwen2_5_vl
  processor_id: null
  torch_dtype: float16
  attn_implementation: sdpa
  trust_remote_code: true
  load_in_4bit: true
  load_in_8bit: false
  min_pixels: 200704
  max_pixels: 602112
lora:
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  bias: none
  task_type: CAUSAL_LM
  target_modules: null
  modules_to_save: null
data:
  dataset_name: cord
  dataset_id: naver-clova-ix/cord-v2
  dataset_subset: null
  image_size: 768
  max_seq_length: 768
  train_split: train
  val_split: validation
  test_split: test
  val_fraction: 0.1
  test_fraction: 0.1
  cache_dir: data/cache
  raw_dir: data/raw
  processed_dir: data/processed
  num_workers: 0
  augment: true
  max_train_samples: null
  max_eval_samples: 100
training:
  output_dir: outputs/qwen2_5vl-cord-lora-benchmark
  num_train_epo

## 6. Train — and survive a disconnect

800 receipts × 2 epochs at effective batch 8 is **200 optimizer steps**; budget roughly
1.5–3 hours on a free T4. The session will very likely drop before that.

That is fine. The cell below looks for the newest `checkpoint-*` under the output dir (which
now lives on Drive) and passes it to `--resume`. **After a disconnect: re-run the notebook
from the top and run this cell again** — it continues from the last checkpoint instead of
starting over. Checkpoints are written every 25 steps, so a death costs at most ~12% of an
epoch.

A checkpoint only counts as resumable if it contains `trainer_state.json`, which the HF
`Trainer` writes *after* the weights and optimizer state. A directory without it was killed
mid-write, so we skip it and fall back to the previous one rather than resuming from a
half-written checkpoint.

In [7]:
# --- 6. Train with automatic resume ----------------------------------------
import re
import subprocess
import sys
from pathlib import Path


def run_streaming(cmd: list[str]) -> int:
    """Run ``cmd``, streaming its output into the cell, and return the exit code.

    We do not use ``!cmd`` here because the bang-magic discards the exit status,
    and a training or evaluation step that failed must never look like one that
    succeeded.
    """
    print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
    return proc.wait()


def latest_checkpoint(output_dir: Path) -> Path | None:
    """Newest complete ``checkpoint-<step>`` dir, or None for a fresh run."""
    candidates = []
    for path in output_dir.glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if not (path.is_dir() and match):
            continue
        if not (path / "trainer_state.json").exists():
            print(f"skipping incomplete checkpoint (no trainer_state.json): {path}")
            continue
        candidates.append((int(match.group(1)), path))
    if not candidates:
        return None
    return max(candidates)[1]


resume_from = latest_checkpoint(OUTPUT_DIR) if OUTPUT_DIR.exists() else None
cmd = [sys.executable, "-m", "training.train", "--config", CONFIG_PATH]
if resume_from is not None:
    print(f"RESUMING from {resume_from}")
    cmd += ["--resume", str(resume_from)]
else:
    print("No checkpoint found — starting a fresh run.")

rc = run_streaming(cmd)
if rc != 0:
    raise RuntimeError(
        f"training.train exited {rc}. If the session was merely disconnected, re-run the "
        f"notebook from the top and run this cell again: it will resume from the newest "
        f"checkpoint under {OUTPUT_DIR}."
    )
print("\nTraining finished.")

No checkpoint found — starting a fresh run.
$ /usr/bin/python3 -m training.train --config configs/benchmark_cord.yaml
2026-09-14 11:21:21.080791: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
<frozen runpy>:130: RuntimeWarning: 'training.train' found in sys.modules after import of package 'training', but prior to execution of 'training.train'; this may result in unpredictable behaviour
[11:21:28] INFO     Compute: device=cuda (Tesla T4) | dtype=float16 | gpus=1 |  
                    mem=14.6 GB                                                 
           INFO     Building model Qwen/Qwen2.5-VL-3B-Instruct (type=qwen2_5_vl)
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will b

In [8]:
# --- 7. Verify the adapter exists ------------------------------------------
# research.compare exits non-zero on a missing adapter rather than silently
# publishing a base-only file, but finding out here is cheaper than finding out
# after a 200-sample evaluation has been queued up.
!ls -la {ADAPTER_DIR}

adapter_config = ADAPTER_DIR / "adapter_config.json"
assert adapter_config.exists(), (
    f"No adapter at {ADAPTER_DIR}. Training did not complete — re-run the training cell "
    f"(it resumes) and do NOT run the comparison until this passes."
)
weights = sorted(
    p.name for p in ADAPTER_DIR.iterdir() if p.suffix in {".safetensors", ".bin"}
)
assert weights, f"{ADAPTER_DIR} has a config but no adapter weights — the save was truncated."
print(f"\nOK: adapter_config.json + weights {weights}")

total 132521
drwx------ 2 root root      4096 Sep 14 12:18 .
drwx------ 6 root root      4096 Sep 14 12:44 ..
-rw------- 1 root root      3973 Sep 14 12:18 adapter_config.json
-rw------- 1 root root 119801528 Sep 14 12:18 adapter_model.safetensors
-rw------- 1 root root       605 Sep 14 12:18 added_tokens.json
-rw------- 1 root root      1049 Sep 14 12:18 chat_template.json
-rw------- 1 root root   1671853 Sep 14 12:18 merges.txt
-rw------- 1 root root       575 Sep 14 12:18 preprocessor_config.json
-rw------- 1 root root      5101 Sep 14 12:18 README.md
-rw------- 1 root root       613 Sep 14 12:18 special_tokens_map.json
-rw------- 1 root root      5824 Sep 14 12:18 tokenizer_config.json
-rw------- 1 root root  11422064 Sep 14 12:18 tokenizer.json
-rw------- 1 root root   2776833 Sep 14 12:18 vocab.json

OK: adapter_config.json + weights ['adapter_model.safetensors']


## 8. Run the comparison — this is the published artifact

One command evaluates **both arms on the identical fixed sample list** and writes
`reports/results.json` (plus `comparison.csv` / `.md` / `.json` and a bar chart).

* `--split validation+test --max-samples 200` pools CORD's two eval splits into the single
  200-item list described at the top. The config's `max_eval_samples: 100` deliberately
  governs only the cheap in-training eval loop, so it is overridden here on the CLI — the
  published *n* is 200 and `results.json` records it.
* A missing or unloadable adapter is a **hard error (exit 2)** with no artifact written. The
  old behaviour — downgrade to base-only, write `comparison.*`, exit 0 — is exactly how a
  one-model file ends up published as a two-model result. If you genuinely want a baseline,
  ask for it explicitly with `--base-only`.
* Expect roughly 20–45 minutes: 200 samples × 2 arms × up to 512 new tokens, greedy.

This is the same invocation as `make results` (identical config, adapter, split and *n*);
the notebook spells the flags out so you can see exactly what was run.

In [ ]:
# --- 8. Base vs LoRA on the fixed 200-sample eval set ----------------------
RESULTS_JSON = Path("reports/results.json")

cmd = [
    sys.executable, "-m", "research.compare",
    "--config", CONFIG_PATH,
    "--adapter", str(ADAPTER_DIR),
    "--split", "validation+test",
    "--max-samples", "200",
    "--results-json", str(RESULTS_JSON),
]
rc = run_streaming(cmd)
if rc != 0:
    raise RuntimeError(
        f"research.compare exited {rc}. Exit 2 means the adapter could not be loaded or "
        f"evaluated; NOTHING was written, which is the intended behaviour. Fix the adapter "
        f"and re-run — do not publish a partial result."
    )

assert RESULTS_JSON.exists(), "compare returned 0 but wrote no results.json — investigate."
print("\n================ reports/results.json ================")
print(RESULTS_JSON.read_text(encoding="utf-8"))

$ /usr/bin/python3 -m research.compare --config configs/benchmark_cord.yaml --adapter outputs/qwen2_5vl-cord-lora-benchmark/adapter --split validation+test --max-samples 200 --results-json reports/results.json
<frozen runpy>:130: RuntimeWarning: 'research.compare' found in sys.modules after import of package 'research', but prior to execution of 'research.compare'; this may result in unpredictable behaviour
[12:44:52] INFO     Compute: device=cuda (Tesla T4) | dtype=float16 | gpus=1 |  
                    mem=14.6 GB                                                 
[12:44:57] INFO     Loading naver-clova-ix/cord-v2 split=validation limit=200   
[12:44:59] INFO     CORD validation: 100 samples                                
           INFO     Split validation -> 100 samples (cap=200)                   
           INFO     Loading naver-clova-ix/cord-v2 split=test limit=200         
[12:45:00] INFO     CORD test: 100 samples                                      
           INFO     Sp

In [ ]:
# --- 9. Copy the artifacts back to Drive ------------------------------------
# reports/ lives on the ephemeral VM disk (it is a git-tracked directory, so it is
# deliberately NOT symlinked to Drive). Copy the outputs across before the session
# can drop: re-running a 200-sample comparison to recover a 6 KB JSON file is a
# painful way to learn this.
import datetime
import shutil

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
snapshot = REPORTS_DIR / stamp        # immutable per-run snapshot
snapshot.mkdir(parents=True, exist_ok=True)

wanted = ["results.json", "comparison.csv", "comparison.md", "comparison.json",
          "comparison_bars.png"]
copied = []
for name in wanted:
    src = Path("reports") / name
    if not src.exists():
        print(f"(absent, skipped) {src}")
        continue
    shutil.copy2(src, snapshot / name)   # timestamped history
    shutil.copy2(src, REPORTS_DIR / name)  # convenient "latest"
    copied.append(name)

print(f"\nCopied {copied}")
print(f"  snapshot : {snapshot}")
print(f"  latest   : {REPORTS_DIR}")
assert "results.json" in copied, "results.json was not produced — nothing to publish."

## 10. Publish the numbers

The notebook's job ends at `reports/results.json`. Publishing is four steps, and none of
them involves typing a number:

1. **Download the artifact.** Grab it from Drive (`visiondoc-runs/reports/results.json`) or
   straight out of Colab:

   ```python
   from google.colab import files
   files.download("reports/results.json")
   ```

2. **Commit it to the repo** at `reports/results.json`. It is the canonical, machine-readable
   record of the run — metrics, bootstrap CIs, *n*, the decoding signature, and the
   environment/provenance block.

   ```bash
   git add reports/results.json
   git commit -m "eval: CORD-v2 base vs LoRA, n=200 (field-F1 primary, bootstrap 95% CI)"
   ```

3. **Regenerate the README table from that JSON:**

   ```bash
   make readme          # == python scripts/update_readme.py --write
   git add README.md
   git commit -m "docs: regenerate results table from reports/results.json"
   ```

   `update_readme.py` rewrites only the block between `<!-- EVAL:BEGIN -->` and
   `<!-- EVAL:END -->`, and that block is a pure function of the results file — no
   timestamps, no latency, no GPU name — so it renders identically on any machine.

4. **Let CI enforce it.** `make readme-check` (== `python scripts/update_readme.py --check`)
   re-renders the block and fails with a diff if the committed table differs by one
   character. **A hand-edited or hand-tuned number cannot be merged.** If the table ever
   needs to change, the only way is to run a real evaluation and commit its `results.json`.

### If the result is disappointing

Publish it anyway, with the explanation. A small delta, an overlapping confidence interval,
or the fine-tune losing to the base model are all real, reportable outcomes — and with
*n* = 200 and bootstrap CIs on the table, a reader can see for themselves whether the
difference means anything. Quietly re-rolling the run until the numbers look better is the
one thing this pipeline exists to prevent.